# Golpe Zero: pipeline de IA generativa com RAG local
FIAP Global Solution 2026.2, CitizenTech. Disciplina Generative AI for Engineering, Prof. Francisco Elânio Bezerra.

Este caderno mostra o pipeline do projeto, na ordem:

1. Carga da base oficial: 158 trechos com fonte, URL e licença.
2. Busca vetorial manual: embeddings `all-MiniLM-L6-v2` e similaridade de cosseno em NumPy.
3. Recall@1, Recall@3 e limiar, com resultado salvo em `avaliacao/recall_busca.json`.
4. Etapa 1 do agente decisor: portão de domínio e piso vetorial.
5. Agente completo com o Qwen 2.5:3B local (Ollama), com saída em JSON.

Conjuntos de avaliação:
* `avaliacao/test_cases.json`: 25 casos (15 golpes, 5 mensagens bancárias legítimas e 5 fora de domínio), todos com gabarito. Os golpes têm `chunks_esperados`.
* `avaliacao/test_cases_stress.json`: 10 casos adversariais (4 golpes sem marca de banco, 3 mensagens legítimas de banco com urgência e 3 fora de domínio com palavras financeiras). Rodam no benchmark completo (`avaliacao/benchmark.py`, 35 casos).

In [1]:
import json
import time
from pathlib import Path
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

# Diretórios base do projeto
ROOT_DIR = Path("..").resolve()
BASE_JSON = ROOT_DIR / "base" / "chunks_oficiais.json"
TEST_CASES_JSON = ROOT_DIR / "avaliacao" / "test_cases.json"

print(f"Diretório raiz: {ROOT_DIR}")
print(f"Existe base de chunks? {BASE_JSON.exists()}")
print(f"Existe dataset de testes? {TEST_CASES_JSON.exists()}")

golpe-zero-genai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1115 08:32:23.250000 4119 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Diretório raiz: golpe-zero-genai
Existe base de chunks? True
Existe dataset de testes? True


### 1. Carregando a Base Curada de Chunks Oficiais (Critério 02)

In [2]:
with open(BASE_JSON, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total de chunks carregados: {len(chunks)}")
print("\nExemplo do primeiro chunk:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False))

Total de chunks carregados: 158

Exemplo do primeiro chunk:
{
  "id": "chunk_certbr_001",
  "fonte": "CERT.br - Cartilha de Segurança para a Internet (Golpes: Não se Deixe Enganar)",
  "url": "https://cartilha.cert.br/fasciculos/#golpes",
  "licenca": "CC BY-NC-SA 4.0",
  "tipo_golpe": "engenharia_social",
  "conteudo": "DESCONFIE. INFORME-SE. VERIFIQUE. G olpistas estão sempre criando novos truques para enganar e tirar vantagem. Fique atento e não caia nessa. Veja aqui como identificar sinais de golpes."
}


### 2. Motor de Álgebra Linear: Similaridade de Cosseno Manual (Critério 03)

$$\text{Sim}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|} = \frac{\sum_{i} A_i B_i}{\sqrt{\sum A_i^2} \times \sqrt{\sum B_i^2}}$$

In [3]:
def similaridade_cosseno_v2(vetor_a: np.ndarray, vetor_b:np.ndarray) -> float:

    #que porra de produto escalar é esse, ja ouvi esse nome antes
    produto_escalar  = np.dot(vetor_a, vetor_b)
    #acho que entendi, isso é a gente iniciar um tensor, fodase vou chamar de tensor sim e vai se fuder voce KKKKKKK, pq aqui nor quer usar o tensorflow em breve pra umas gracinha
    #então ele faz alguma merda, que merda é essa, normalizar, normalizar linalg, pra que essa merda gente, da onde isso
    norma_a = np.linalg.norm(vetor_a) #que merda é esse dot e esse linalg, e esse norm ai nao sei nada disso
    norma_b = np.linalg.norm(vetor_b)

    if norma_a == 0.0 or norma_b == 0.0:
        return 0.0

        #aqui ele ve se ta dando 0, pq ai nao tem nada a ver com nada, faz sentido

    return float(produto_escalar / (norma_a * norma_b))

#por que produto escalar, pulta quil paril, e digo mais, eu detesto visualizar forma com notação matematica, eu acho a maior imbecilidade do mundo tu usar os mesmo simbolos que nego usava na epoca de Euler e Gauss, e na epoca tinha motivo de economizar papel e ter uma padronização, hoje tu tem python pra escrever formula de maneira muito mais intuitiva, ficar escrevendo formula de simbolozinho é coisa dos fetichista academico, pronto falei e vai todo mundo se fuder, aqui a gente quer economizar banda larga mental pra resolver a treta, não ficar complciando as coisas


#ok ,ai tem o teste com veotres sinteticos aqui, pq 1d, uma dimensão ? parece ter 3 krl]
#ai ele monta o array com tais numeros, deixa eu piratear aqui pera ai

v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([1.0, 2.0, 3.0])
v3 = np.array([-1.0, -2.0, -3.0])

print(f"Cosseno entre vetores idênticos (esperado 1.0): {similaridade_cosseno_v2(v1, v2):.4f}")
print(f"Cosseno entre vetores opostos (esperado -1.0): {similaridade_cosseno_v2(v1, v3):.4f}")

Cosseno entre vetores idênticos (esperado 1.0): 1.0000
Cosseno entre vetores opostos (esperado -1.0): -1.0000


In [4]:
def similaridade_cosseno(vetor_a: np.ndarray, vetor_b: np.ndarray) -> float:
    """Calcula a similaridade manual usando apenas operações primitivas de NumPy."""
    produto_escalar = np.dot(vetor_a, vetor_b)
    norma_a = np.linalg.norm(vetor_a)
    norma_b = np.linalg.norm(vetor_b)
    
    if norma_a == 0.0 or norma_b == 0.0:
        return 0.0
        
    return float(produto_escalar / (norma_a * norma_b))

# Teste com vetores sintéticos 1D
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([1.0, 2.0, 3.0])
v3 = np.array([-1.0, -2.0, -3.0])
print(f"Cosseno entre vetores idênticos (esperado 1.0): {similaridade_cosseno(v1, v2):.4f}")
print(f"Cosseno entre vetores opostos (esperado -1.0): {similaridade_cosseno(v1, v3):.4f}")

#resumindo, ele faz um vezes o outro dividido por alguma porra, explica melhor isso ai rapá

Cosseno entre vetores idênticos (esperado 1.0): 1.0000
Cosseno entre vetores opostos (esperado -1.0): -1.0000


### 3. Vetorização da Base com `all-MiniLM-L6-v2`

In [5]:
print("Carregando modelo de embeddings leve (all-MiniLM-L6-v2)...")
modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")

textos_chunks = [c["conteudo"] for c in chunks]
print(f"Codificando {len(textos_chunks)} trechos...")
matriz_embeddings = np.array(modelo_emb.encode(textos_chunks, show_progress_bar=True))

print(f"Formato da Matriz de Chunks: {matriz_embeddings.shape} -> (Total Chunks, Dimensões)")

Carregando modelo de embeddings leve (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4317.20it/s]

Codificando 158 trechos...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:  20%|██        | 1/5 [00:01<00:06,  1.54s/it]

Batches:  40%|████      | 2/5 [00:01<00:02,  1.44it/s]

Batches:  80%|████████  | 4/5 [00:01<00:00,  3.18it/s]

Batches: 100%|██████████| 5/5 [00:01<00:00,  2.64it/s]

Formato da Matriz de Chunks: (158, 384) -> (Total Chunks, Dimensões)


### 4. Função de Busca Manual dos Top K Chunks

In [6]:
def buscar_top_k(mensagem: str, k: int = 3):
    vetor_msg = modelo_emb.encode(mensagem, show_progress_bar=False)
    
    # Calcula similaridade contra cada chunk individualmente
    scores = []
    for idx, vetor_chunk in enumerate(matriz_embeddings):
        sim = similaridade_cosseno(vetor_msg, vetor_chunk)
        scores.append((sim, idx))
        
    # Ordena do maior para o menor score
    scores.sort(key=lambda x: x[0], reverse=True)
    
    resultados = []
    for score, idx in scores[:k]:
        item = dict(chunks[idx])
        item["score"] = round(score, 4)
        resultados.append(item)
    return resultados

# Teste com um golpe clássico
consulta_teste = "Meu cartão Bradesco foi clonado e disseram para eu cortar ao meio e entregar pro motoboy"
top_chunks = buscar_top_k(consulta_teste, k=3)

print(f"Consulta: \"{consulta_teste}\"\n")
for r in top_chunks:
    print(f"Score: {r['score']:.4f} | Fonte: {r['fonte']}")
    print(f"Conteúdo: {r['conteudo'][:140]}...\n")

Consulta: "Meu cartão Bradesco foi clonado e disseram para eu cortar ao meio e entregar pro motoboy"

Score: 0.5435 | Fonte: Febraban Antifraudes - Golpe da maquininha quebrada
Conteúdo: Também podem simular uma ligação de uma falsa central informando problemas na cobrança no cartão e solicitam os dados de cobrança da vítima....

Score: 0.5429 | Fonte: Febraban Antifraudes - Golpe da maquininha quebrada
Conteúdo: Também podem simular uma ligação de uma falsa central informando problemas na cobrança no cartão e solicitam os dados de cobrança da vítima....

Score: 0.5267 | Fonte: Febraban Antifraudes - Golpe da troca de cartão
Conteúdo: Eles se aproveitam de alguma dificuldade sua no terminal eletrônico para pegar rapidamente o seu cartão e depois devolver um que não é seu, ...



### 5. Teste da receita de bolo com urgência
A palavra "urgência" aproxima a mensagem dos chunks de golpe do falso familiar (score 0.5886, acima do piso de 0.35). Quem recusa essa mensagem é o portão de domínio da etapa 1, que não encontra nenhum termo financeiro nela.

In [7]:
msg_bolo = "Vó, me manda a receita do bolo de fubá com urgência! Preciso começar a fazer agora a sobremesa da festa que os convidados já estão chegando! Me passa logo por favor!"
retorno_bolo = buscar_top_k(msg_bolo, k=3)

print(f"Score Máximo retornado para o Bolo de Fubá: {retorno_bolo[0]['score']}")
print(f"Chunk associado: {retorno_bolo[0]['fonte']}")
print(f"Trecho: {retorno_bolo[0]['conteudo']}")

Score Máximo retornado para o Bolo de Fubá: 0.5886
Chunk associado: CERT.br - Cartilha de Segurança para a Internet (Golpes: Não se Deixe Enganar)
Trecho: REDOBRE A ATENÇÃO COM TEMAS EM DESTAQUE e stá todo mundo falando de um assunto? Golpistas aproveitam temas em alta para atrair vítimas. Fique atento a assuntos como: - eleições - promoções (ex: Black Friday) - tragédias (ex: enchentes, pandemia) - grandes eventos (ex: Copa do Mundo) - impostos (ex: IPVA, Imposto de Renda) Promoções, doações, débitos ou pendências? - só em canais oficiais! Oferta muito boa? - lembre-se: ”quando a esmola é demais, o santo desconfia” 90% DESCONTO


### 5.1 Avaliação Formal de Recuperação: Recall@1, Recall@3 e Calibração do Limiar (Critério 03)

A função `avaliar_metricas` de `src/vector_search.py` executa a bateria formal sobre os **25 casos principais** (`avaliacao/test_cases.json`: **15 golpes, 5 legítimas e 5 fora de domínio**) usando o mesmo motor vetorial do benchmark e da API. Para cada caso de golpe, o gabarito de recuperação são os `chunks_esperados` anotados em `base/chunks_oficiais.json`:

* **Recall@1:** fração de golpes cujo chunk mais similar (Top 1) pertence ao gabarito;
* **Recall@3:** fração de golpes com ao menos um chunk do gabarito no Top 3;
* **Limiar:** o ponto médio entre o maior score fora de domínio e o menor de golpe é 0.48, mas as distribuições se sobrepõem e esse corte descartaria 3 golpes reais. Adota-se um piso de $\theta = 0.35$ (menor score de golpe, 0.3958, menos 0.05), porque deixar passar um golpe custa mais ao idoso do que acionar a LLM. A recusa de mensagens fora de domínio fica a cargo do portão lexical da Etapa 1 (`src/llm_engine.py`).

O resultado medido é persistido em **`avaliacao/recall_busca.json`** (hits por caso, médias/mín/máx de score por categoria e limiar). O split de estresse (10 casos, `avaliacao/test_cases_stress.json`) é avaliado no benchmark comparativo das 3 arquiteturas.

In [8]:
import sys
import pandas as pd

# Permite que o pacote src do repositório seja importável a partir do notebook
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.vector_search import MotorBuscaVetorial, avaliar_metricas

# Motor oficial (mesma classe usada pelo benchmark e pela API)
motor = MotorBuscaVetorial()

# Avaliação formal sobre os 25 casos principais; persiste avaliacao/recall_busca.json
recall = avaliar_metricas(motor)

# Tabela de hits por caso de golpe (gabarito: chunks_esperados)
df_golpes = pd.DataFrame([c for c in recall["casos"] if c["categoria_esperada"] == "golpe"])
display(df_golpes[["id", "tipo_golpe_esperado", "score_top1", "top1_chunk_id", "hit_top1", "hit_top3"]])

# Resumo oficial: recalls medidos e limiar sugerido
resumo = pd.DataFrame([
    {"Métrica": "Recall@1 (%)", "Valor": recall["recall_at_1_pct"]},
    {"Métrica": "Recall@3 (%)", "Valor": recall["recall_at_3_pct"]},
    {"Métrica": "Golpes avaliados", "Valor": recall["total_golpes"]},
    {"Métrica": "Limiar ponto médio (não separa as classes)", "Valor": recall["limiar_ponto_medio"]},
    {"Métrica": "Piso adotado (min golpe - 0.05)", "Valor": recall["limiar_sugerido"]},
])
display(resumo)

# Estatísticas de score por categoria (calibração do limiar)
df_scores = pd.DataFrame(recall["scores_por_categoria"]).T
df_scores.index.name = "categoria"
display(df_scores)

[Motor] Carregando modelo de embeddings 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3765.61it/s]

[Motor] Carregando base de conhecimento de 'golpe-zero-genai/base/chunks_oficiais.json'...
[Motor] Vetorizando 158 chunks oficiais...


[Motor] Indexação concluída na memória com sucesso.

INICIANDO AUDITORIA EXPERIMENTAL DO MOTOR VETORIAL (RECALL E THRESHOLD)
[tc_001] Golpe: falsa_central_atendiment | Score Top 1: 0.4979 | R@1: HIT | R@3: HIT
[tc_002] Golpe: falso_whatsapp           | Score Top 1: 0.5966 | R@1: MISS | R@3: MISS
[tc_003] Golpe: falso_motoboy            | Score Top 1: 0.6762 | R@1: HIT | R@3: HIT
[tc_004] Golpe: maquininha_quebrada_pres | Score Top 1: 0.5387 | R@1: MISS | R@3: MISS
[tc_005] Golpe: pix_errado_comprovante_f | Score Top 1: 0.6517 | R@1: HIT | R@3: HIT
[tc_006] Golpe: mao_fantasma_acesso_remo | Score Top 1: 0.5513 | R@1: MISS | R@3: HIT
[tc_007] Golpe: falso_boleto             | Score Top 1: 0.6586 | R@1: MISS | R@3: MISS
[tc_008] Golpe: falso_emprestimo         | Score Top 1: 0.5883 | R@1: HIT | R@3: HIT
[tc_009] Golpe: phishing                 | Score Top 1: 0.4721 | R@1: MISS | R@3: MISS
[tc_010] Golpe: falso_investimento       | Score Top 1: 0.5503 | R@1: MISS | R@3: HIT
[tc_011] Golpe:

[tc_024] Fora Domínio: pergunta esportiva s     | Score Máx: 0.5141
[tc_025] Fora Domínio: dica doméstica fora      | Score Máx: 0.5269

RESULTADOS OFICIAIS DE RECUPERAÇÃO VETORIAL (Critério 03)
• Total de casos de golpe avaliados : 15
• Recall@1 (Top 1 exato)           : 33.33% (5/15)
• Recall@3 (Presente no Top 3)     : 46.67% (7/15)
----------------------------------------------------------------------
CALIBRAÇÃO EMPÍRICA DO LIMIAR MATEMÁTICO (THRESHOLD):
 - Golpes Reais       -> Média: 0.5501 | Mínimo: 0.3958 | Máximo: 0.6762
 - Casos Legítimos    -> Média: 0.6335 | Mínimo: 0.5735 | Máximo: 0.7443
 - Fora de Domínio    -> Média: 0.5282 | Mínimo: 0.4988 | Máximo: 0.5646

=> PONTO MEDIO (nao separa as classes, classes se sobrepoem): 0.48
=> PISO ADOTADO (min golpe 0.3958 - 0.05): 0.35
   (Consulta com score < 0.35 ativa o veredito compulsório 'não consta')

[Artefato] Métricas de recuperação persistidas em: golpe-zero-genai/avaliacao/recall_busca.json


,id,tipo_golpe_esperado,score_top1,top1_chunk_id,hit_top1,hit_top3
0,tc_001,falsa_central_atendimento,0.4979,chunk_febraban_002,True,True
1,tc_002,falso_whatsapp,0.5966,chunk_febraban_002,False,False
2,tc_003,falso_motoboy,0.6762,chunk_febraban_021,True,True
3,tc_004,maquininha_quebrada_presente,0.5387,chunk_febraban_031,False,False
4,tc_005,pix_errado_comprovante_falso,0.6517,chunk_certbr_032,True,True
5,tc_006,mao_fantasma_acesso_remoto,0.5513,chunk_febraban_068,False,True
6,tc_007,falso_boleto,0.6586,chunk_sspsp_003,False,False
7,tc_008,falso_emprestimo,0.5883,chunk_febraban_051,True,True
8,tc_009,phishing,0.4721,chunk_proconsp_001,False,False
9,tc_010,falso_investimento,0.5503,chunk_bcb_014,False,True


,Métrica,Valor
0,Recall@1 (%),33.33
1,Recall@3 (%),46.67
2,Golpes avaliados,15.00
3,Limiar ponto médio (não separa as classes),0.48
4,Piso adotado (min golpe - 0.05),0.35


,media,minimo,maximo
categoria,,,
golpe,0.5501,0.3958,0.6762
legitimo,0.6335,0.5735,0.7443
fora_de_dominio,0.5282,0.4988,0.5646


### 6. Etapa 1 do Agente Decisor (portão de domínio + piso vetorial)
O código de produção está em `src/llm_engine.py`. Aqui ele é chamado diretamente, sem reimplementação.

In [9]:
from src.llm_engine import PipelineGolpeZero, mensagem_no_dominio, THRESHOLD_CORTE

pipeline = PipelineGolpeZero(motor)

def triagem_agente_decisor(mensagem: str):
    """Mostra o que a Etapa 1 decide para a mensagem, sem chamar a LLM."""
    score_max = buscar_top_k(mensagem, k=3)[0]["score"]
    no_dominio = mensagem_no_dominio(mensagem)
    barrada = (not no_dominio) or score_max < THRESHOLD_CORTE
    return {"termo_de_dominio": no_dominio, "score_max": round(score_max, 4),
            "piso": THRESHOLD_CORTE, "recusada_na_etapa_1": barrada}

print("Teste 1 (Bolo de Fubá):", triagem_agente_decisor(msg_bolo))
print("Teste 2 (Falso Motoboy):", triagem_agente_decisor(consulta_teste))

Teste 1 (Bolo de Fubá): {'termo_de_dominio': False, 'score_max': 0.5886, 'piso': 0.35, 'recusada_na_etapa_1': True}
Teste 2 (Falso Motoboy): {'termo_de_dominio': True, 'score_max': 0.5435, 'piso': 0.35, 'recusada_na_etapa_1': False}


### 7. Agente Decisor completo com o Qwen 2.5:3B local (Ollama)

In [10]:
import json

for rotulo, msg in (("Bolo de Fubá", msg_bolo), ("Falso Motoboy", consulta_teste)):
    res = pipeline.predizer_agente_decisor(msg)
    print(f"--- {rotulo} | {res['_etapa_decisao']} | {res['_latencia_seg']}s")
    print(json.dumps({k: v for k, v in res.items() if not k.startswith("_")}, indent=2, ensure_ascii=False))

--- Bolo de Fubá | Etapa 1 (Portao de dominio e piso vetorial) | 0.008s
{
  "veredito": "nao_consta",
  "tipo_golpe": "nao_se_aplica",
  "sinais_encontrados": [
    "Nenhum termo de banco, pagamento ou compra na mensagem"
  ],
  "fonte_oficial": "Base Oficial Febraban/BCB/CERT.br/Procon-SP",
  "proximo_passo": "Mensagem não categorizada como fraude nas bases oficiais conhecidas. Mantenha cautela e contate seu banco se tiver dúvidas."
}


--- Falso Motoboy | Etapa 2 (LLM RAG Fundamentado) | 18.669s
{
  "veredito": "golpe",
  "tipo_golpe": "falso_motoboy",
  "sinais_encontrados": [
    "motoboy"
  ],
  "fonte_oficial": "Febraban",
  "proximo_passo": "DESLIGUE o telefone imediatamente e ligue para o número atrás do seu cartão"
}
